# Stage 1 Training (Colab)

Make sure **Runtime -> Change runtime type -> GPU (T4 / L4 / A100)** is selected.

Open `TB.ipynb` in a separate tab to view TensorBoard. Independent kernel, will not block training.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/autoTest
!ls

## 2. Install torchinfo
`torchinfo` is the only extra dep we need (used by `hyperparameter_dump` for model summaries). PyTorch + CUDA come from Colab's preinstalled versions — no version pinning needed.

In [ ]:
!pip install -q torchinfo

## 3. Environment check
Sanity check that CUDA is visible and a GPU was assigned (T4 / L4 / A100 etc.). PyTorch / CUDA version numbers will be whatever Colab currently ships.

In [ ]:
!nvidia-smi

import sys
import torch

print(f'Python      : {sys.version.split()[0]}')
print(f'PyTorch     : {torch.__version__}')
print(f'CUDA build  : {torch.version.cuda}')
print(f'CUDA avail  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device      : {torch.cuda.get_device_name(0)}')

## 4. Anti-idle (reduce disconnects)
Colab disconnects after ~90 minutes of idle. Open browser **F12 -> Console** and paste this snippet; it simulates a click on the connect button every 60 seconds:

```javascript
function ClickConnect() {
    document.querySelector('colab-connect-button')
        ?.shadowRoot?.querySelector('#connect')?.click();
    console.log('anti-idle:', new Date().toLocaleTimeString());
}
setInterval(ClickConnect, 60000);
```

This does NOT bypass Google's hard limits (free tier ~12h/day, GPU quota). It only reduces idle-based disconnects.

## 5. Set up local workspace and background sync
**Why**: Colab's Drive mount is a FUSE writeback layer. Writes only hit a local buffer; they are **not guaranteed to be uploaded to Drive cloud**. When the runtime disconnects, the buffer dies with it and any unsaved progress is lost (this is exactly why `.pth` files did not appear on Drive earlier).

**Strategy**:
- Symlink `autoTest_pytorch/` to Drive (read-only access to source, no copy needed)
- Copy `models/` from Drive to local SSD; training reads and writes locally (fast + survives FUSE quirks)
- Background rsync local -> Drive every 3 minutes so Drive has a recent snapshot

Worst case on disconnect: lose the last 3 minutes of progress.

In [ ]:
import os
import subprocess

WORKSPACE = '/content/workspace'
DRIVE_ROOT = '/content/drive/MyDrive/autoTest'

os.makedirs(WORKSPACE, exist_ok=True)

# Symlink source tree (read-only from training's perspective, no need to copy).
code_link = f'{WORKSPACE}/autoTest_pytorch'
if not os.path.exists(code_link):
    os.symlink(f'{DRIVE_ROOT}/autoTest_pytorch', code_link)
    print(f'symlink: {code_link} -> {DRIVE_ROOT}/autoTest_pytorch')
else:
    print(f'(symlink already exists: {code_link})')

# Copy models from Drive to local SSD (training will write here).
# To force a fresh copy, manually `!rm -rf /content/workspace/models` first.
local_models = f'{WORKSPACE}/models'
if not os.path.exists(local_models):
    print('Initial copy: Drive/models -> /content/workspace/models ...')
    subprocess.run(
        ['cp', '-r', f'{DRIVE_ROOT}/models', local_models],
        check=True,
    )
    print('done')
else:
    print(f'(reusing existing {local_models})')

print(f'\nWorkspace ready at {WORKSPACE}')
!ls -la /content/workspace

In [ ]:
import os
import subprocess

# Background rsync: local workspace -> Drive every SYNC_INTERVAL seconds.
# No --delete, so an empty/partial local will not wipe Drive.
SYNC_INTERVAL = 180
PID_FILE = '/content/sync.pid'
LOG_FILE = '/content/sync.log'

# Kill any previous sync process from earlier cell runs to avoid duplicates.
if os.path.exists(PID_FILE):
    try:
        old_pid = int(open(PID_FILE).read().strip())
        subprocess.run(['kill', str(old_pid)], check=False)
        print(f'killed previous sync PID={old_pid}')
    except (ValueError, FileNotFoundError):
        pass

sync_cmd = f'''
while true; do
  sleep {SYNC_INTERVAL}
  rsync -a /content/workspace/models/ /content/drive/MyDrive/autoTest/models/ 2>/dev/null
  echo "[$(date +%H:%M:%S)] synced local -> Drive"
done
'''

sync_proc = subprocess.Popen(
    ['bash', '-c', sync_cmd],
    stdout=open(LOG_FILE, 'a'),
    stderr=subprocess.STDOUT,
)

with open(PID_FILE, 'w') as f:
    f.write(str(sync_proc.pid))

print(f'Background rsync running, PID={sync_proc.pid}')
print(f'  interval : every {SYNC_INTERVAL}s')
print(f'  log      : !tail -f {LOG_FILE}')
print(f'  stop     : !kill {sync_proc.pid}')

## 6. Run training
Launch from the workspace, not from Drive. `train_stage1_simple.py` writes to the relative path `./models/stage1_transformer/`, so once cwd is the workspace it writes to local SSD.

Interrupting is fine (stop button or Runtime -> Interrupt). After interrupting, **always run the final-sync cell below**.

In [ ]:
%cd /content/workspace
!python autoTest_pytorch/train_stage1_simple.py

## 7. Final sync (MUST run after interrupt or completion)
Background rsync only fires every 3 minutes, so the last chunk of progress is still on local SSD when training stops. **Only close Colab after you see `final sync done`**, otherwise the local SSD disappears with the runtime.

In [ ]:
!rsync -av /content/workspace/models/ /content/drive/MyDrive/autoTest/models/ \
    && echo '=== final sync done ==='